### Importing the data from hugging face

In [11]:
from datasets import load_dataset
import pandas as pd
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from tqdm.notebook import tqdm
from sacrebleu.metrics import CHRF, BLEU

# CONFIG
model_name = "facebook/m2m100_418M"
lang_pairs = [
    # Original pairs from your screenshot
    ('af', 'zu'), ('ar', 'fr'), ('be', 'ru'), ('cs', 'sk'), ('de', 'hr'), ('de', 'hu'), ('el', 'tr'),
    ('fr', 'sw'), ('hi', 'bn'), ('hi', 'mr'), ('hr', 'cs'), ('hr', 'hu'), ('hr', 'sk'), ('hr', 'sr'),
    ('it', 'de'), ('it', 'fr'), ('nl', 'de'), ('nl', 'fr'), ('ro', 'de'), ('ro', 'hu'), ('ro', 'hy'),
    ('ro', 'ru'), ('ro', 'tr'), ('ro', 'uk'), ('uk', 'ru'),

    # Added English, French, German pairs (no identity pairs)
    ('en', 'fr'), ('fr', 'en'),
    ('en', 'de'), ('de', 'en'),
    ('fr', 'de'), ('de', 'fr')
]


lang_map = {
    'af': 'afr', 'zu': 'zul', 'ar': 'ara', 'fr': 'fra', 'be': 'bel', 'ru': 'rus', 'cs': 'ces',
    'sk': 'slk', 'de': 'deu', 'hr': 'hrv', 'hu': 'hun', 'el': 'ell', 'tr': 'tur', 'sw': 'swh',
    'hi': 'hin', 'bn': 'ben', 'mr': 'mar', 'sr': 'srp', 'it': 'ita', 'nl': 'nld', 'ro': 'ron',
    'hy': 'hye', 'uk': 'ukr'
}

drop_columns = ['has_image', 'has_hyperlink', 'URL', 'domain', 'topic']
num_samples = 997

tqdm.pandas()
chrf = CHRF(word_order=2)
bleu = BLEU(effective_order=True)

# Load model
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# --- Load FLORES dataset ---
def load_language_df(lang_code):
    dataset = load_dataset("gsarti/flores_101", lang_map[lang_code])
    df = pd.DataFrame(dataset['dev'])[:num_samples]
    return df.drop(columns=drop_columns)

### Translate using M2M-100

In [12]:
# --- Translate a sentence ---
def translate(sentence, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(sentence, return_tensors="pt")
    generated = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang)
    )
    return tokenizer.decode(generated[0], skip_special_tokens=True)


# --- Full pipeline for one language pair ---
def process_lang_pair(src, tgt):
    print(f"\n🔄 Translating from {src} → {tgt}")
    df_src = load_language_df(src)
    df_tgt = load_language_df(tgt)

    df_src['translation'] = df_src['sentence'].progress_apply(lambda x: translate(x, src, tgt))
    
    df_src['chrf2_score'] = [
        chrf.sentence_score(h, [r]).score for h, r in zip(df_src['translation'], df_tgt['sentence'])
    ]
    df_src['spbleu_score'] = [
        bleu.sentence_score(h, [r]).score for h, r in zip(df_src['translation'], df_tgt['sentence'])
    ]


    return df_src

# --- Process all pairs ---
results = {}

for src, tgt in lang_pairs:
    try:
        df_result = process_lang_pair(src, tgt)
        results[f"{src}_to_{tgt}"] = df_result
    except Exception as e:
        print(f"⚠️ Failed for {src}-{tgt}: {e}")


🔄 Translating from af → zu


0000.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

...

0000.parquet:   0%|          | 0.00/169k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

  0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Let's see if it hallucinates

In [ ]:
for name, df in results.items():
    print(f"\n--- {name} ---")
    print(df.head(10))
